# Zeitansage_Agent: ein Agent mit einem Tool

Ein **Agent** ist ein Sprachmodell, das selbst entscheidet, ob es ein
**Tool** braucht. Ein Tool ist eine normale Python-Funktion, die der Agent aufrufen darf.
Unser Agent hat ein Tool: Es sagt die Uhrzeit in der Schweiz und wie lange der Kurstag noch dauert.

## Vorbereitung (einmal)
1. **Kopie speichern:** *Datei → Kopie in Drive speichern*. Sonst gehen deine Änderungen verloren.
2. **Secrets hinterlegen:** links auf das **Schlüssel-Symbol** (Secrets) klicken und
   drei Secrets anlegen (Werte von Moodle). Bei allen **Notebook-Zugriff** einschalten.

   | Name | Bedeutung |
   |---|---|
   | `BASE_URL` | Adresse des Anbieters, der die Sprachmodelle betreibt |
   | `API_KEY` | dein Zugangsschlüssel |
   | `MODEL` | welches Sprachmodell standardmässig verwendet wird |

   So stehen Key und Adresse nie im Notebook selbst.

## Jedes Mal
Colab löscht nach einer Pause alles. Führe die Zellen darum **von oben nach unten** aus
(Play-Knopf links an jeder Zelle, oder *Laufzeit → Alle ausführen*).

> **Tipp für den Code-Assistenten Gemini:** Schreib in deine Anfrage dazu:
> *«Nutze Google ADK. Als Modell immer
> `LiteLlm(model="openai/" + MODEL, api_base=BASE_URL, api_key=API_KEY)`.
> Halte den Code so einfach wie möglich, Kommentare auf Deutsch.»*

## 1. Einrichten

In [ ]:
# Google ADK (Agent Development Kit) und LiteLLM installieren. Dauert ca. 1 Minute.
# LiteLLM = Adapter, damit ADK Sprachmodelle verschiedener Anbieter nutzen kann.
!pip install -q "google-adk>=2.15,<3" litellm

In [ ]:
# Die drei Secrets holen (Schlüssel-Symbol links).
from google.colab import userdata

BASE_URL = userdata.get("BASE_URL")  # Adresse des Anbieters
API_KEY = userdata.get("API_KEY")    # Zugangsschlüssel
MODEL = userdata.get("MODEL")        # Standard-Modell für alle Agenten

print("Secrets geladen. Standard-Modell:", MODEL)

## 2. Das Tool

Wichtig ist der **Docstring** (der Text in `""" """`): Daran erkennt das Sprachmodell, wofür das Tool gut ist.
`tool_context` gibt ADK automatisch mit. Darüber kann das Tool den Ablauf beeinflussen (siehe Demo unten).

In [ ]:
import math
from datetime import datetime
from zoneinfo import ZoneInfo

from google.adk.tools import ToolContext


def aktuelle_uhrzeit(tool_context: ToolContext) -> dict:
    """Gibt die aktuelle Uhrzeit in der Schweiz zurück, mit einem Hinweis zum Kurstag."""
    jetzt = datetime.now(ZoneInfo("Europe/Zurich"))  # Colab läuft in UTC, darum die Zeitzone
    kursende = jetzt.replace(hour=17, minute=15, second=0, microsecond=0)

    if jetzt < kursende:
        rest_minuten = math.ceil((kursende - jetzt).total_seconds() / 60)
        stunden = rest_minuten // 60
        minuten = rest_minuten % 60
        hinweis = f"CAS Kurstag: Dann musst du noch maximal {stunden} Stunden und {minuten} Minuten durchhalten."
    else:
        hinweis = "Feierabend!"

    ansage = f"Es ist {jetzt:%H:%M} Uhr. {hinweis}"

    # tool_context.actions.skip_summarization = True

    # --- DEMO: Entferne das # vor der Zeile oben, dann wird das Tool-Ergebnis
    # NICHT an das Sprachmodell geschickt.
    # Normalerweise geht das Tool-Ergebnis zurück an das Sprachmodell, und es
    # formuliert daraus die Antwort (zweite Modellrunde).
    # Ohne # zeigt ADK das Ergebnis direkt an, und das Sprachmodell sieht es nie.
    # -------------------------------------------------------------------------

    return {"status": "ok", "ansage": ansage}

## 3. Der Agent

- `model`: welches Sprachmodell denkt. `LiteLlm` verbindet ADK mit dem Anbieter aus den Secrets.
  `"openai/"` vor dem Modellnamen heisst: Der Anbieter spricht dieselbe «Sprache» wie OpenAI
  (das können die meisten). Für ein anderes Modell nur in diesem Agenten ersetzt du `MODEL`
  durch einen Modellnamen in Anführungszeichen, z.B. `"openai/" + "anbieter/modellname"`.
- `description`: kurze Beschreibung, wichtig, wenn später mehrere Agenten zusammenarbeiten
- `instruction`: die Anweisung an das Modell (der «Systemprompt»)
- `tools`: die Funktionen, die der Agent aufrufen darf

In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

root_agent = Agent(
    name="Zeitansage_Agent",
    model=LiteLlm(model="openai/" + MODEL, api_base=BASE_URL, api_key=API_KEY),
    description="Sagt die aktuelle Uhrzeit an und wie lange der Kurstag noch dauert.",
    instruction=(
        "Du bist ein freundlicher Assistent in einem Weiterbildungskurs. "
        "Antworte kurz und auf Deutsch. "
        "Wenn nach Datum, Uhrzeit oder Kursende gefragt wird, rufe das Tool "
        "aktuelle_uhrzeit auf und gib dessen Feld 'ansage' mit einer kleinen total lustigen Variation wieder."
    ),
    tools=[aktuelle_uhrzeit],
)

## 4. Den Agenten fragen

Der **Runner** führt den Agenten aus und merkt sich das Gespräch.
Angezeigt wird nur die Antwort. Alles, was dabei passiert ist, landet in `ereignisse`
und wird in den nächsten beiden Zellen genauer angeschaut.
Ändere die Frage und führe die Zelle nochmal aus.

In [ ]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=root_agent)
ereignisse = await runner.run_debug("Wie spät ist es?")

## Welche Tools wurden aufgerufen?

`ereignisse` ist die Liste von allem, was beim Ausführen passiert ist. Jedes **Ereignis**
(Event) hat einen Absender (`author`) und Teile (`parts`): Text, Tool-Aufruf oder Tool-Ergebnis.
Diese Zelle zeigt nur die Tool-Aufrufe und ihre Ergebnisse.

In [ ]:
for ereignis in ereignisse:
    if not ereignis.content or not ereignis.content.parts:
        continue  # Ereignis ohne Inhalt, z.B. nur eine Änderung im State
    for teil in ereignis.content.parts:
        if teil.function_call:
            print(f"{ereignis.author} ruft auf: {teil.function_call.name}({teil.function_call.args})")
        if teil.function_response:
            print(f"Ergebnis: {teil.function_response.response}\n")

## Alle Details

So sieht ADK die Ereignisse intern. Du findest hier z.B. welches Modell geantwortet hat
(`model_version`), wie viele **Tokens** (Textstücke) gelesen und geschrieben wurden
(`usage_metadata`, danach wird bezahlt) und die Änderungen am State (`state_delta`).

In [ ]:
ereignisse

## 5. Ausprobieren

1. **Eine Frage ohne Tool:** Frag zum Beispiel *«Erzähl mir einen Witz.»* Ruft der Agent das Tool auf?
2. **Ein Gespräch:** Der Runner merkt sich, was vorher gesagt wurde, bis du die Runner-Zelle
   wieder ausführst. Führe die Zelle unten mehrmals mit verschiedenen Fragen aus.
   Danach zeigen die Tool- und die Detail-Zelle oben die Ereignisse dieser neuen Frage.
3. **Demo `skip_summarization`:** Entferne im Tool das `#` vor `tool_context.actions...`.
   Führe danach die Tool-Zelle, die Agent-Zelle und die Frage-Zelle **nochmal aus**.
   Die Antwort-Zelle bleibt jetzt leer. Wo steht die Uhrzeit? Wer hat den Text formuliert?

In [ ]:
ereignisse = await runner.run_debug("Und wann ist Feierabend?")